In [57]:
import re, torch, statistics, random, json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm
from peft import PeftModel
from difflib import SequenceMatcher
from datasets import Dataset
from tqdm import tqdm

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_DIR = "./lora-output/final_trained"

In [4]:
test_ds = Dataset.from_pandas((pd.read_json("dataset/test_dataset.jsonl", lines=True)))

# load tokenizer from the adapter, not base model
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# load base weights
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="cuda:0"
)

# load lora-adapter weights
model = PeftModel.from_pretrained(
    model,
    ADAPTER_DIR,
    torch_dtype=torch.bfloat16,
)

model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
def strip_end_token(text: str, token: str = "<|im_end|>") -> str:
    return text.replace(token, "")

def extract_fix_block(text: str) -> str:

    m = re.search(r"(### FIX:.*)$", text, flags=re.DOTALL)
    if not m:
        return ""

    block = m.group(1)

    stop = re.search(r"\bassistant\b", block)
    if stop:
        return block[:stop.start()].rstrip()

    return block.strip()

def normalize(s: str) -> str:
        s = s.strip()
        s = re.sub(r"\s+", " ", s)  # collapse all whitespace sequences
        return s

def get_similarity(true_text: str, generated_text: str) -> float:
    
    t1 = normalize(true_text)
    t2 = normalize(generated_text)

    return SequenceMatcher(None, t1, t2).ratio()

In [55]:
def ask_model(example: dict):
    prompt = tokenizer.apply_chat_template(
        [example["messages"][0]],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True) # greedy decoding


def evaluate_response(ground_truth: str, answer: str):
    generated_fix = extract_fix_block(answer)
    true_fix = extract_fix_block(strip_end_token(ground_truth))

    similarity = get_similarity(true_fix, generated_fix)

    return generated_fix, true_fix, similarity

In [56]:
# Debugging by testing a random example 
i = random.randrange(len(test_ds)) 
example = test_ds[i]

answer = ask_model(example)
ground_truth = example["messages"][1]["content"]

generated_fix, true_fix, similarity = evaluate_response(ground_truth, answer)

print(f"Example Index: {i}")
print(f"Similarity: {similarity}")
print(f"True Fix: \n{true_fix}")
print(f"Gen Fix: \n{generated_fix}")

Example Index: 898
Similarity: 1.0
True Fix: 
### FIX:
### ## REPLACE:
        connect comp_a_01b6.p to comp_distractor_8d01.p;

## WITH:
        connect comp_a_01b6.p to comp_b_0f6a.p;
Gen Fix: 
### FIX:
### ## REPLACE:
        connect comp_a_01b6.p to comp_distractor_8d01.p;

## WITH:
        connect comp_a_01b6.p to comp_b_0f6a.p;


In [ ]:
# Testing entire dataset (this can take very long time)

results = {
    "truth": [],
    "answer": [],
    "generated": [],
    "similarity": []
}

sim_values = []

pbar = tqdm(test_ds, desc="Evaluating", dynamic_ncols=True)

for example in pbar:

    answer = ask_model(example)
    ground_truth = example["messages"][1]["content"]

    generated_fix, true_fix, similarity = evaluate_response(ground_truth, answer)

    results["truth"].append(true_fix)
    results["answer"].append(answer)
    results["generated"].append(generated_fix)
    results["similarity"].append(similarity)

    sim_values.append(similarity)
    mean_sim = statistics.mean(sim_values)

    pbar.set_postfix({
        "last_sim": f"{similarity:.4f}",
        "mean_sim": f"{mean_sim:.4f}"
    })

with open("test_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Testing complete. Results saved to test_results.json")

Evaluating:   0%|          | 1/950 [00:07<2:00:30,  7.62s/it, last_sim=1.0000, mean_sim=1.0000]
